# NB02 — Evaluation harness

**Input:** NB00's consolidated panels and NB01's feature datasets, through `container.panel_store()` and `container.feature_store()`. They are byte-identical to the freeze `data/backups/thesis_2026-10-30/`, which the spec's numbers were measured on; do not re-ingest or re-run NB01 while this notebook is in use.

**Output (so far):** Checkpoint 1 — the four baselines scored under Protocol A, checked against the spec's table.

Spec: `docs/specs/nb02-evaluation-harness.md`. Plan: `docs/plans/nb02-evaluation-harness.md`. This notebook only orchestrates; every rule lives in `app.services.evaluation`.

In [1]:
from itertools import product, starmap

import pandas as pd

from app.domain import TARGET_VARIABLES, TargetVariable
from app.injections import configure_container
from app.services.evaluation import Baseline, BaselineForecaster, ForecastTask
from app.services.features import default_arm

container = configure_container()
panels = container.panel_store().load_consolidated()
feature_store = container.feature_store()
harness = container.evaluation_harness()
len(panels)

2026-11-05 09:55:12.467 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-05 09:55:12.470 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


60

## Checkpoint 1 (A1): the baselines on Protocol A

Each baseline goes through the same path as a model: `BaselineForecaster` → `harness.evaluate`. The spec's table covers h = 1 and h = 4.

In [2]:
HORIZONS = (1, 4)


def score(
    variable: TargetVariable, horizon: int, baseline: Baseline
) -> pd.DataFrame:
    task = ForecastTask(variable=variable, arm=default_arm(variable), horizon=horizon)
    evaluation = harness.evaluate(
        BaselineForecaster(baseline, panels=panels, task=task),
        feature_store.read(variable.value, horizon),
        panels=panels,
        task=task,
    )
    return evaluation.metrics.pooled.assign(
        variable=variable.value,
        horizon=horizon,
        baseline=baseline.value,
        baseline_to_beat=evaluation.baseline_to_beat.value,
    )


scores = pd.concat(
    list(starmap(score, product(TARGET_VARIABLES, HORIZONS, Baseline))),
    ignore_index=True,
)
scores.pivot_table(
    index=["variable", "horizon"], columns="baseline", values="mae"
).round(4)

baseline                         drift   naive  seasonal_naive  \
variable               horizon                                   
ebitda_usd_m           1        0.0150  0.0150          0.0164   
                       4        0.0171  0.0172          0.0172   
free_cash_flow_usd_m   1        0.0271  0.0269          0.0200   
                       4        0.0215  0.0209          0.0209   
operating_income_usd_m 1        0.0150  0.0150          0.0163   
                       4        0.0171  0.0170          0.0170   
revenue_usd_m          1        0.0810  0.0813          0.0906   
                       4        0.0953  0.0906          0.0906   

baseline                        seasonal_naive_growth  
variable               horizon                         
ebitda_usd_m           1                       0.0179  
                       4                       0.0258  
free_cash_flow_usd_m   1                       0.0237  
                       4                       0.0333  
operating_income_usd_m 1                       0.0182  
                       4                       0.0262  
revenue_usd_m          1                       0.0574  
                       4                       0.1223

The spec's table ("Two baselines, and neither dominates"), to four decimals. Naive is in the spec for revenue only.

In [3]:
SPEC_MAE = pd.DataFrame(
    [
        ("revenue_usd_m", 1, "seasonal_naive", 0.0906),
        ("revenue_usd_m", 1, "seasonal_naive_growth", 0.0574),
        ("revenue_usd_m", 1, "naive", 0.0813),
        ("revenue_usd_m", 4, "seasonal_naive", 0.0906),
        ("revenue_usd_m", 4, "seasonal_naive_growth", 0.1223),
        ("revenue_usd_m", 4, "naive", 0.0906),
        ("operating_income_usd_m", 1, "seasonal_naive", 0.0163),
        ("operating_income_usd_m", 1, "seasonal_naive_growth", 0.0182),
        ("operating_income_usd_m", 4, "seasonal_naive", 0.0170),
        ("operating_income_usd_m", 4, "seasonal_naive_growth", 0.0262),
        ("ebitda_usd_m", 1, "seasonal_naive", 0.0164),
        ("ebitda_usd_m", 1, "seasonal_naive_growth", 0.0179),
        ("ebitda_usd_m", 4, "seasonal_naive", 0.0172),
        ("ebitda_usd_m", 4, "seasonal_naive_growth", 0.0258),
        ("free_cash_flow_usd_m", 1, "seasonal_naive", 0.0200),
        ("free_cash_flow_usd_m", 1, "seasonal_naive_growth", 0.0237),
        ("free_cash_flow_usd_m", 4, "seasonal_naive", 0.0209),
        ("free_cash_flow_usd_m", 4, "seasonal_naive_growth", 0.0333),
    ],
    columns=["variable", "horizon", "baseline", "spec_mae"],
)

comparison = SPEC_MAE.merge(scores, on=["variable", "horizon", "baseline"])
comparison["harness_mae"] = comparison["mae"].round(4)
comparison["matches"] = comparison["harness_mae"] == comparison["spec_mae"]
comparison.filter(
    items=["variable", "horizon", "baseline", "spec_mae", "harness_mae", "matches"]
)

,variable,horizon,baseline,spec_mae,harness_mae,matches
0,revenue_usd_m,1,seasonal_naive,0.0906,0.0906,True
1,revenue_usd_m,1,seasonal_naive_growth,0.0574,0.0574,True
2,revenue_usd_m,1,naive,0.0813,0.0813,True
3,revenue_usd_m,4,seasonal_naive,0.0906,0.0906,True
4,revenue_usd_m,4,seasonal_naive_growth,0.1223,0.1223,True
5,revenue_usd_m,4,naive,0.0906,0.0906,True
6,operating_income_usd_m,1,seasonal_naive,0.0163,0.0163,True
7,operating_income_usd_m,1,seasonal_naive_growth,0.0182,0.0182,True
8,operating_income_usd_m,4,seasonal_naive,0.0170,0.0170,True
9,operating_income_usd_m,4,seasonal_naive_growth,0.0262,0.0262,True


In [4]:
assert len(comparison) == len(SPEC_MAE), "a spec row has no harness score"
assert comparison["matches"].all(), comparison.loc[~comparison["matches"]]

# The spec's test-row counts for revenue ("Origins with a known target").
revenue = scores[scores["variable"] == "revenue_usd_m"]
revenue_rows = revenue.groupby("horizon")["rows"].first()
assert revenue_rows.to_dict() == {1: 956, 4: 958}, revenue_rows

# The baseline to beat is the spec's bold one: with growth for revenue at h = 1.
to_beat = scores.groupby(["variable", "horizon"])["baseline_to_beat"].first()
expected_to_beat = {
    key: "seasonal_naive_growth" if key == ("revenue_usd_m", 1) else "seasonal_naive"
    for key in to_beat.index
}
assert to_beat.to_dict() == expected_to_beat, to_beat